# Chapter 11: Multi-Modal Perception Agents

**Book:** *30 Agents Every AI Engineer Must Build*
**Author:** Imran Ahmad
**Publisher:** Packt Publishing

---

## Overview

This notebook implements three families of multi-modal perception agents:

1. **Part 1 — Vision-Language Agents:** Architecture, Visual Question Answering, and integration patterns.
2. **Part 2 — Audio Processing Agents:** Speech recognition with filler removal and voice sentiment analysis.
3. **Part 3 — Physical World Sensing Agents:** Smart building management with event detection, control loops, and sensor fusion.

Each agent follows the **Sense → Model → Plan → Act** loop described in the chapter. The notebook runs in **Simulation Mode** (mock backends) or **Live Mode** (real model inference) — auto-detected at startup.

> **Ref:** Chapter 11 — Technical Requirements

In [ ]:
# =============================================================================
# Cell 2: Environment Detection & Configuration
# Ref: Chapter 11 — Technical Requirements
# Author: Imran Ahmad
# =============================================================================
# Detects GPU availability, loads .env secrets, and sets SIMULATION_MODE.
# If any prerequisite is missing, the notebook gracefully falls back to
# Simulation Mode using high-fidelity mock backends.
# =============================================================================

import os
import sys
from pathlib import Path

# --- Load .env if present ---
try:
    from dotenv import load_dotenv
    env_path = Path(".env")
    if env_path.exists():
        load_dotenv(dotenv_path=env_path)
except ImportError:
    pass  # dotenv not installed — will use Simulation Mode

# --- Detect hardware & credentials ---
gpu_available = False
torch_available = False
try:
    import torch
    torch_available = True
    gpu_available = torch.cuda.is_available()
except ImportError:
    pass

hf_token = os.environ.get("HUGGINGFACE_TOKEN", "")
has_token = bool(hf_token) and hf_token != "your_hugging_face_token_here"

# --- Determine mode ---
SIMULATION_MODE = not (gpu_available and has_token and torch_available)

# --- Collect reasons ---
reasons = []
if not torch_available:
    reasons.append("torch not installed")
if not gpu_available and torch_available:
    reasons.append("no CUDA GPU detected")
if not has_token:
    reasons.append("no HUGGINGFACE_TOKEN in .env")

print(f"SIMULATION_MODE = {SIMULATION_MODE}")
if reasons:
    print(f"  Reasons: {', '.join(reasons)}")
else:
    print("  Live Mode — all prerequisites satisfied.")

In [ ]:
# =============================================================================
# Cell 3: Color-Coded Mode Banner
# Ref: Chapter 11 — Technical Requirements
# Author: Imran Ahmad
# =============================================================================

from agent_logger import AgentLogger

logger = AgentLogger(name="Chapter11")

if SIMULATION_MODE:
    logger.info(
        "Simulation Mode active. All agents will use mock backends. "
        "No GPU or API keys required. Output is chapter-sourced."
    )
else:
    logger.success(
        "Live Mode active. Agents will use real model inference. "
        "GPU and Hugging Face token detected."
    )

---

# Part 1: Vision-Language Agents

## Architecture of Vision-Language Agents
**Ref:** Chapter 11 — Architecture of Vision-Language Agents

Vision-language models like LLaVA 1.5 combine a **vision encoder** (CLIP ViT-L/14) with a **language model** (Vicuna 7B) through an **alignment mechanism** — a learned linear projection that maps visual features into the language model's embedding space.

The key insight is that the alignment mechanism does not require the vision encoder and language model to be trained jointly. Instead, a lightweight projection layer is trained on image-text pairs while both the encoder and decoder remain frozen (or are lightly fine-tuned).

**Agent loop:** Sense (camera/image input) → Model (VLM inference) → Plan (parse answer) → Act (return structured response)

In [ ]:
# =============================================================================
# Cell 5: Import Layer — Mock or Real Backends
# Ref: Chapter 11 — Technical Requirements
# Author: Imran Ahmad
# =============================================================================
# Conditional import: in Simulation Mode, we load MockVLM and MockProcessor.
# In Live Mode, we load the real Hugging Face transformers classes.
# The agent class code below is IDENTICAL in both modes — only the
# backend object injected at construction time changes.
# =============================================================================

import numpy as np
from PIL import Image, ImageDraw, ImageFont

if SIMULATION_MODE:
    from mock_backends import MockVLM, MockProcessor
    logger.info("Vision backends: MockVLM, MockProcessor loaded.")
else:
    from transformers import LlavaForConditionalGeneration, AutoProcessor
    import torch
    logger.info("Vision backends: LlavaForConditionalGeneration, AutoProcessor loaded.")

In [ ]:
# =============================================================================
# Cell 6: VisionQuestionAnsweringAgent
# Ref: Chapter 11 — Building a Vision QA Agent
# Author: Imran Ahmad
# =============================================================================

from agent_logger import graceful_fallback
from dataclasses import dataclass
from typing import Optional


@dataclass
class VQAResult:
    """Result of a visual question-answering query.

    Ref: Building a Vision QA Agent.
    Author: Imran Ahmad

    Attributes:
        question: The original question text.
        answer:   The model's natural-language answer.
        success:  Whether the inference completed without error.
    """
    question: str
    answer: str
    success: bool = True


class VisionQuestionAnsweringAgent:
    """Agent that answers natural-language questions about images.

    Ref: Chapter 11 — Building a Vision QA Agent.
    Author: Imran Ahmad

    Uses a vision-language model (LLaVA 1.5 in Live Mode, MockVLM in
    Simulation Mode) with an alignment mechanism that projects visual
    features into the language model's embedding space.

    Args:
        model:     A VLM instance (real or mock) with .generate() and .decode().
        processor: A processor instance (real or mock) for image+text packaging.
        device:    Target device string ('cuda', 'cpu', or 'mock').
    """

    def __init__(self, model, processor, device: str = "mock") -> None:
        self.model = model
        self.processor = processor
        self.device = device
        self.logger = AgentLogger(name="VisionQA")
        self.logger.info("Initializing Vision-Language Agent...")
        mode_label = "Simulation Mode" if SIMULATION_MODE else "Live Mode"
        self.logger.success(f"Agent ready ({mode_label})")

    @graceful_fallback(
        fallback_value=VQAResult(question="", answer="Error — see log", success=False),
        chapter_ref="Building a Vision QA Agent",
    )
    def answer_question(self, image: Image.Image, question: str) -> VQAResult:
        """Answer a natural-language question about the given image.

        Ref: Building a Vision QA Agent — inference pipeline.
        Author: Imran Ahmad

        Sense:  Receive image + question.
        Model:  Encode via processor, run VLM generate.
        Plan:   Decode token IDs to text.
        Act:    Return structured VQAResult.

        Args:
            image:    A PIL Image to analyze.
            question: Natural-language question about the image.

        Returns:
            VQAResult with the model's answer.
        """
        self.logger.info(f"answer_question called: \"{question[:60]}\"")

        # Sense + Model: processor packages image+text, model generates
        prompt = f"USER: <image>\n{question}\nASSISTANT:"
        inputs = self.processor(text=prompt, images=image, return_tensors="pt")

        if not SIMULATION_MODE:
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

        output_ids = self.model.generate(**inputs)

        # Plan: decode tokens to natural language
        answer = self.model.decode(output_ids[0], skip_special_tokens=True)

        # Act: return structured result
        self.logger.success(
            f"answer_question completed. "
            f"CoT reasoning extracted. Answer length: {len(answer)} chars."
        )
        return VQAResult(question=question, answer=answer, success=True)

In [ ]:
# =============================================================================
# Cell 7: Programmatic Test Image — Synthetic Workspace Scene
# Ref: Chapter 11 — Building a Vision QA Agent
# Author: Imran Ahmad
# =============================================================================
# Generates a simple synthetic workspace image so the notebook is
# self-contained — no external image downloads required.
# =============================================================================

def create_test_workspace_image(width: int = 640, height: int = 480) -> Image.Image:
    """Generate a synthetic workspace image for VQA demos.

    Ref: Building a Vision QA Agent — test fixtures.
    Author: Imran Ahmad

    Creates a simple scene with colored rectangles representing:
    desk (brown), monitors (dark gray), keyboard (light gray),
    lamp (yellow), mug (red), and two person silhouettes.

    Args:
        width:  Image width in pixels.
        height: Image height in pixels.

    Returns:
        PIL Image with a synthetic workspace scene.
    """
    img = Image.new("RGB", (width, height), color=(240, 240, 235))
    draw = ImageDraw.Draw(img)

    # Desk surface
    draw.rectangle([50, 300, 590, 380], fill=(139, 90, 43))

    # Left monitor
    draw.rectangle([120, 120, 280, 290], fill=(50, 50, 55))
    draw.rectangle([130, 130, 270, 270], fill=(70, 130, 180))  # screen glow

    # Right monitor
    draw.rectangle([320, 120, 480, 290], fill=(50, 50, 55))
    draw.rectangle([330, 130, 470, 270], fill=(70, 130, 180))

    # Keyboard
    draw.rectangle([200, 310, 400, 345], fill=(180, 180, 180))

    # Desk lamp (upper-left)
    draw.rectangle([70, 80, 90, 180], fill=(90, 90, 90))   # stand
    draw.ellipse([55, 55, 105, 90], fill=(255, 220, 80))     # bulb

    # Coffee mug (right of keyboard)
    draw.rectangle([430, 310, 460, 350], fill=(180, 40, 40))
    draw.ellipse([428, 305, 462, 320], fill=(180, 40, 40))

    # Person 1 — seated at desk (silhouette)
    draw.ellipse([280, 200, 320, 240], fill=(100, 100, 100))   # head
    draw.rectangle([285, 240, 315, 310], fill=(100, 100, 100)) # torso

    # Person 2 — standing near whiteboard (background)
    draw.ellipse([520, 60, 555, 95], fill=(120, 120, 120))     # head
    draw.rectangle([525, 95, 550, 200], fill=(120, 120, 120))  # torso

    # Whiteboard on back wall
    draw.rectangle([490, 30, 600, 180], fill=(250, 250, 250))
    draw.rectangle([492, 32, 598, 178], outline=(180, 180, 180))

    # Sticky notes on left monitor bezel
    for i, color in enumerate([(255, 255, 100), (100, 255, 150), (255, 180, 100)]):
        draw.rectangle([122 + i * 18, 115, 136 + i * 18, 130], fill=color)

    return img


# Generate and save test image
test_image = create_test_workspace_image()
os.makedirs("assets", exist_ok=True)
test_image.save("assets/sample_workspace.png")
logger.success("Test image generated: assets/sample_workspace.png (640x480)")
test_image

In [ ]:
# =============================================================================
# Cell 8: Initialize Vision-Language Agent
# Ref: Chapter 11 — Architecture of Vision-Language Agents
# Author: Imran Ahmad
# =============================================================================
# SIMULATION_MODE gate: inject mock or real backends at construction time.
# The VisionQuestionAnsweringAgent class is identical regardless of mode.
# =============================================================================

if SIMULATION_MODE:
    vlm_model = MockVLM(model_id="mock-llava-1.5-7b")
    vlm_processor = MockProcessor(model_id="mock-llava-1.5-7b")
    vlm_device = "mock"
else:
    model_id = "llava-hf/llava-1.5-7b-hf"
    vlm_model = LlavaForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        token=hf_token,
    )
    vlm_processor = AutoProcessor.from_pretrained(model_id, token=hf_token)
    vlm_device = "cuda"

vqa_agent = VisionQuestionAnsweringAgent(
    model=vlm_model,
    processor=vlm_processor,
    device=vlm_device,
)

In [ ]:
# =============================================================================
# Cell 9: Vision Agent Demo — Describe Query
# Ref: Chapter 11 — Building a Vision QA Agent (scenario: "describe")
# Author: Imran Ahmad
# =============================================================================

result_describe = vqa_agent.answer_question(
    image=test_image,
    question="Describe this workspace in detail.",
)
print(f"\nQuestion: {result_describe.question}")
print(f"Answer:   {result_describe.answer}")
print(f"Success:  {result_describe.success}")

In [ ]:
# =============================================================================
# Cell 10: Vision Agent Demo — Count Query
# Ref: Chapter 11 — Building a Vision QA Agent (scenario: "count")
# Author: Imran Ahmad
# =============================================================================

result_count = vqa_agent.answer_question(
    image=test_image,
    question="Count the people visible in this image.",
)
print(f"\nQuestion: {result_count.question}")
print(f"Answer:   {result_count.answer}")
print(f"Success:  {result_count.success}")

In [ ]:
# =============================================================================
# Cell 11: Vision Agent Demo — Spatial Relationship Query
# Ref: Chapter 11 — Integration Patterns (scenario: "spatial")
# Author: Imran Ahmad
# =============================================================================

result_spatial = vqa_agent.answer_question(
    image=test_image,
    question="Describe the spatial relationship of objects on the desk.",
)
print(f"\nQuestion: {result_spatial.question}")
print(f"Answer:   {result_spatial.answer}")
print(f"Success:  {result_spatial.success}")

In [ ]:
# =============================================================================
# Cell 12: Vision Agent Demo — Error / Fallback Scenario
# Ref: Chapter 11 — Integration Patterns (error handling)
# Author: Imran Ahmad
# =============================================================================
# Passing None as the image triggers the @graceful_fallback decorator.
# The agent logs a RED error and returns a safe fallback VQAResult
# instead of crashing the notebook.
# =============================================================================

result_error = vqa_agent.answer_question(
    image=None,  # Intentional: triggers graceful fallback
    question="Describe this image.",
)
print(f"\nQuestion: {result_error.question}")
print(f"Answer:   {result_error.answer}")
print(f"Success:  {result_error.success}")
print("\n(The RED error above is intentional — demonstrating @graceful_fallback)")

---

# Part 2: Audio Processing Agents

## Architecture of Audio Processing Agents
**Ref:** Chapter 11 — Architecture of Audio Processing Agents

Audio processing agents convert raw acoustic signals into structured information through a multi-stage pipeline:

1. **Acoustic Feature Extraction** — Raw waveforms are transformed into mel-spectrograms, capturing frequency content over time.
2. **Speech Recognition (ASR)** — Models like Whisper decode spectrograms into timestamped text segments.
3. **Post-Processing** — Filler word removal (clean mode) or preservation (verbatim mode) depending on the use case.
4. **Sentiment Analysis** — Prosodic features (pitch, speech rate, energy) are mapped to emotional states using a Valence-Arousal-Dominance (VAD) model.

**Agent loop:** Sense (microphone/audio input) → Model (Whisper ASR + prosodic extraction) → Plan (clean/annotate transcript) → Act (return structured result)

In [ ]:
# =============================================================================
# Cell 14: Import Layer — Audio Backends
# Ref: Chapter 11 — Architecture of Audio Processing Agents
# Author: Imran Ahmad
# =============================================================================

from dataclasses import dataclass, field
from enum import Enum
from typing import List
import re

if SIMULATION_MODE:
    from mock_backends import MockWhisperBackend, MockTranscriptionSegment
    logger.info("Audio backends: MockWhisperBackend loaded.")
else:
    # In Live Mode, you would import whisper or transformers pipeline here.
    # from transformers import pipeline
    logger.info("Audio backends: Live Whisper pipeline loaded.")

In [ ]:
# =============================================================================
# Cell 15: Audio Processing Data Structures
# Ref: Chapter 11 — Building a Speech Recognition Agent
# Author: Imran Ahmad
# =============================================================================

class TranscriptionMode(Enum):
    """Transcription output mode.

    Ref: Building a Speech Recognition Agent — clean vs. verbatim.
    Author: Imran Ahmad

    CLEAN:    Remove filler words ('um', 'uh', 'like', 'you know').
    VERBATIM: Preserve all speech exactly as spoken.
    """
    CLEAN = "clean"
    VERBATIM = "verbatim"


@dataclass
class TranscriptionSegment:
    """A single timestamped segment of transcribed speech.

    Ref: Building a Speech Recognition Agent.
    Author: Imran Ahmad
    """
    start: float
    end: float
    text: str
    confidence: float = 0.95


@dataclass
class TranscriptionResult:
    """Complete transcription output with metadata.

    Ref: Building a Speech Recognition Agent.
    Author: Imran Ahmad
    """
    segments: List[TranscriptionSegment]
    mode: TranscriptionMode
    full_text: str = ""
    filler_count_removed: int = 0

    def __post_init__(self) -> None:
        if not self.full_text:
            self.full_text = " ".join(seg.text for seg in self.segments)


@dataclass
class ProsodicFeatures:
    """Acoustic prosodic features extracted from speech.

    Ref: Voice Sentiment Analysis — prosodic feature extraction.
    Author: Imran Ahmad

    Attributes:
        pitch_hz:        Fundamental frequency (F0) in Hertz.
        speech_rate_sps: Syllables per second.
        energy_db:       RMS energy in decibels.
    """
    pitch_hz: float
    speech_rate_sps: float
    energy_db: float


@dataclass
class SentimentResult:
    """Voice sentiment analysis output.

    Ref: Voice Sentiment Analysis — VAD model.
    Author: Imran Ahmad
    """
    primary_emotion: str
    confidence: float
    prosodic_features: ProsodicFeatures
    valence: float = 0.0
    arousal: float = 0.0
    dominance: float = 0.0

logger.success("Audio data structures defined: TranscriptionMode, TranscriptionSegment, "
               "TranscriptionResult, ProsodicFeatures, SentimentResult.")

In [ ]:
# =============================================================================
# Cell 16: SpeechRecognitionAgent
# Ref: Chapter 11 — Building a Speech Recognition Agent
# Author: Imran Ahmad
# =============================================================================

# Filler patterns for clean mode
_FILLER_PATTERN = re.compile(
    r"\b(um|uh|er|ah|like|you know|I mean|sort of|kind of)\b",
    re.IGNORECASE,
)


class SpeechRecognitionAgent:
    """Agent that transcribes audio with optional filler word removal.

    Ref: Chapter 11 — Building a Speech Recognition Agent.
    Author: Imran Ahmad

    In CLEAN mode, filler words are stripped from each segment using
    regex pattern matching. In VERBATIM mode, all speech is preserved
    exactly as recognized.

    Args:
        backend: A Whisper-compatible backend (real or mock) with .transcribe().
    """

    def __init__(self, backend) -> None:
        self.backend = backend
        self.logger = AgentLogger(name="SpeechAgent")
        self.logger.info("Initializing Speech Recognition Agent...")
        mode_label = "Simulation Mode" if SIMULATION_MODE else "Live Mode"
        self.logger.success(f"Agent ready ({mode_label})")

    @staticmethod
    def _clean_text(text: str) -> tuple[str, int]:
        """Remove filler words from text.

        Ref: Building a Speech Recognition Agent — filler removal.
        Author: Imran Ahmad

        Args:
            text: Raw transcription text.

        Returns:
            Tuple of (cleaned text, number of fillers removed).
        """
        fillers_found = len(_FILLER_PATTERN.findall(text))
        cleaned = _FILLER_PATTERN.sub("", text)
        cleaned = re.sub(r"\s{2,}", " ", cleaned).strip()
        return cleaned, fillers_found

    @graceful_fallback(
        fallback_value=TranscriptionResult(segments=[], mode=TranscriptionMode.CLEAN),
        chapter_ref="Building a Speech Recognition Agent",
    )
    def transcribe_audio(
        self,
        audio: np.ndarray,
        mode: TranscriptionMode = TranscriptionMode.CLEAN,
        scenario: str = "customer_complaint",
    ) -> TranscriptionResult:
        """Transcribe audio input with optional filler removal.

        Ref: Building a Speech Recognition Agent — inference pipeline.
        Author: Imran Ahmad

        Sense:  Receive audio waveform.
        Model:  Run Whisper ASR to get timestamped segments.
        Plan:   Apply filler removal if CLEAN mode.
        Act:    Return structured TranscriptionResult.

        Args:
            audio:    Audio waveform as numpy array.
            mode:     CLEAN (remove fillers) or VERBATIM (preserve all).
            scenario: Mock scenario key for Simulation Mode routing.

        Returns:
            TranscriptionResult with segments and metadata.
        """
        self.logger.info(f"transcribe_audio called: mode={mode.value}, scenario={scenario}")

        # Sense + Model: get raw segments from backend
        is_clean = (mode == TranscriptionMode.CLEAN)
        raw_segments = self.backend.transcribe(audio, scenario=scenario, clean=is_clean)

        # Plan: convert backend segments to our dataclass
        total_fillers = 0
        result_segments = []
        for seg in raw_segments:
            if mode == TranscriptionMode.CLEAN:
                cleaned_text, n_fillers = self._clean_text(seg.text)
                total_fillers += n_fillers
                result_segments.append(TranscriptionSegment(
                    start=seg.start, end=seg.end,
                    text=cleaned_text, confidence=seg.confidence,
                ))
            else:
                result_segments.append(TranscriptionSegment(
                    start=seg.start, end=seg.end,
                    text=seg.text, confidence=seg.confidence,
                ))

        result = TranscriptionResult(
            segments=result_segments,
            mode=mode,
            filler_count_removed=total_fillers,
        )

        # Act: log success
        if mode == TranscriptionMode.CLEAN:
            self.logger.success(
                f"transcribe_audio completed. {len(result_segments)} segments, "
                f"'um/uh' removed."
            )
        else:
            self.logger.success(
                f"transcribe_audio completed. Verbatim mode, fillers preserved."
            )

        return result

In [ ]:
# =============================================================================
# Cell 17: VoiceSentimentAgent
# Ref: Chapter 11 — Voice Sentiment Analysis
# Author: Imran Ahmad
# =============================================================================

class VoiceSentimentAgent:
    """Agent that detects emotional state from voice prosodic features.

    Ref: Chapter 11 — Voice Sentiment Analysis.
    Author: Imran Ahmad

    Uses a Valence-Arousal-Dominance (VAD) model to map prosodic features
    (pitch, speech rate, energy) to emotional categories. The mapping is
    based on established psychoacoustic research:

    - High pitch + high rate + high energy → angry / frustrated
    - Low pitch + low rate + low energy    → sad / calm
    - High pitch + moderate rate           → happy / excited

    Args:
        backend: A Whisper-compatible backend with .get_prosodic_features().
    """

    # VAD mapping thresholds
    # Ref: Voice Sentiment Analysis — VAD model thresholds
    _VAD_MAP = {
        "angry":   {"pitch_min": 180, "rate_min": 4.5, "energy_min": -20, "valence": -0.7, "arousal": 0.8, "dominance": 0.6},
        "happy":   {"pitch_min": 170, "rate_min": 4.0, "energy_min": -22, "valence": 0.8, "arousal": 0.7, "dominance": 0.5},
        "sad":     {"pitch_min": 0,   "rate_min": 0,   "energy_min": -40, "valence": -0.6, "arousal": -0.5, "dominance": -0.4},
        "calm":    {"pitch_min": 0,   "rate_min": 0,   "energy_min": -35, "valence": 0.3, "arousal": -0.6, "dominance": 0.2},
    }

    def __init__(self, backend) -> None:
        self.backend = backend
        self.logger = AgentLogger(name="SentimentAgent")
        self.logger.info("Initializing Voice Sentiment Agent...")
        mode_label = "Simulation Mode" if SIMULATION_MODE else "Live Mode"
        self.logger.success(f"Agent ready ({mode_label})")

    @graceful_fallback(
        fallback_value=SentimentResult(
            primary_emotion="unknown", confidence=0.0,
            prosodic_features=ProsodicFeatures(0.0, 0.0, 0.0),
        ),
        chapter_ref="Voice Sentiment Analysis",
    )
    def analyze_sentiment(
        self,
        audio: np.ndarray,
        emotion_scenario: str = "angry",
    ) -> SentimentResult:
        """Analyze voice sentiment from prosodic features.

        Ref: Voice Sentiment Analysis — VAD mapping pipeline.
        Author: Imran Ahmad

        Sense:  Receive audio waveform.
        Model:  Extract prosodic features (pitch, rate, energy).
        Plan:   Map features to VAD dimensions, classify emotion.
        Act:    Return structured SentimentResult.

        Args:
            audio:             Audio waveform as numpy array.
            emotion_scenario:  Mock scenario key ('angry', 'calm').

        Returns:
            SentimentResult with emotion classification and VAD scores.
        """
        self.logger.info(f"analyze_sentiment called: scenario={emotion_scenario}")

        # Sense + Model: extract prosodic features
        raw_features = self.backend.get_prosodic_features(audio, emotion=emotion_scenario)
        features = ProsodicFeatures(
            pitch_hz=raw_features["pitch_hz"],
            speech_rate_sps=raw_features["speech_rate_sps"],
            energy_db=raw_features["energy_db"],
        )

        # Plan: classify emotion via VAD mapping
        # Strategy: match the most specific emotion (highest thresholds)
        # that the features satisfy. Specificity = sum of threshold minimums.
        detected_emotion = "calm"  # default fallback
        best_specificity = -999.0

        for emotion, thresholds in self._VAD_MAP.items():
            if (features.pitch_hz >= thresholds["pitch_min"]
                    and features.speech_rate_sps >= thresholds["rate_min"]
                    and features.energy_db >= thresholds["energy_min"]):
                specificity = (thresholds["pitch_min"]
                               + thresholds["rate_min"] * 10
                               + thresholds["energy_min"])
                if specificity > best_specificity:
                    best_specificity = specificity
                    detected_emotion = emotion

        best_confidence = min(0.95, 0.6 + features.pitch_hz / 500)

        vad = self._VAD_MAP[detected_emotion]
        result = SentimentResult(
            primary_emotion=detected_emotion,
            confidence=round(best_confidence, 3),
            prosodic_features=features,
            valence=vad["valence"],
            arousal=vad["arousal"],
            dominance=vad["dominance"],
        )

        # Act: log result
        self.logger.success(
            f"analyze_sentiment completed. Primary emotion: {detected_emotion} "
            f"(confidence: {result.confidence})"
        )

        return result

In [ ]:
# =============================================================================
# Cell 18: Initialize Audio Agents
# Ref: Chapter 11 — Building a Speech Recognition Agent
# Author: Imran Ahmad
# =============================================================================

if SIMULATION_MODE:
    whisper_backend = MockWhisperBackend(model_id="mock-whisper-large-v3")
else:
    # In Live Mode, initialize real Whisper pipeline here
    pass

speech_agent = SpeechRecognitionAgent(backend=whisper_backend)
sentiment_agent = VoiceSentimentAgent(backend=whisper_backend)

In [ ]:
# =============================================================================
# Cell 19: Synthetic Audio Placeholder
# Ref: Chapter 11 — Building a Speech Recognition Agent
# Author: Imran Ahmad
# =============================================================================
# In Simulation Mode, the mock backend ignores the actual audio data and
# routes by scenario key. We create a synthetic sine wave as a placeholder
# to demonstrate the correct input type (numpy array).
# =============================================================================

sample_rate = 16000  # 16 kHz — standard for Whisper
duration_sec = 5.0
t = np.linspace(0, duration_sec, int(sample_rate * duration_sec), endpoint=False)
synthetic_audio = (0.5 * np.sin(2 * np.pi * 440 * t)).astype(np.float32)

logger.info(f"Synthetic audio generated: {len(synthetic_audio)} samples at {sample_rate}Hz "
            f"({duration_sec}s, 440Hz sine wave)")

In [ ]:
# =============================================================================
# Cell 20: Speech Demo — CLEAN Mode (Customer Complaint)
# Ref: Chapter 11 — Building a Speech Recognition Agent (scenario: customer_complaint)
# Author: Imran Ahmad
# =============================================================================
# Demonstrates filler word removal: 'um' and 'uh' are stripped from
# the transcript, producing clean text suitable for downstream NLP.
# =============================================================================

result_clean = speech_agent.transcribe_audio(
    audio=synthetic_audio,
    mode=TranscriptionMode.CLEAN,
    scenario="customer_complaint",
)

print(f"\nMode: {result_clean.mode.value}")
print(f"Segments: {len(result_clean.segments)}")
print(f"Fillers removed: {result_clean.filler_count_removed}")
print(f"\nFull transcript (clean):")
for seg in result_clean.segments:
    print(f"  [{seg.start:.1f}s - {seg.end:.1f}s] {seg.text}")

In [ ]:
# =============================================================================
# Cell 21: Speech Demo — VERBATIM Mode (Meeting Notes)
# Ref: Chapter 11 — Building a Speech Recognition Agent (scenario: meeting_notes)
# Author: Imran Ahmad
# =============================================================================
# Demonstrates verbatim transcription: all filler words ('um', 'uh')
# are preserved, useful for linguistic analysis or meeting archives.
# =============================================================================

result_verbatim = speech_agent.transcribe_audio(
    audio=synthetic_audio,
    mode=TranscriptionMode.VERBATIM,
    scenario="meeting_notes",
)

print(f"\nMode: {result_verbatim.mode.value}")
print(f"Segments: {len(result_verbatim.segments)}")
print(f"\nFull transcript (verbatim):")
for seg in result_verbatim.segments:
    print(f"  [{seg.start:.1f}s - {seg.end:.1f}s] {seg.text}")

In [ ]:
# =============================================================================
# Cell 22: Voice Sentiment Analysis Demo — Frustrated Caller
# Ref: Chapter 11 — Voice Sentiment Analysis (prosodic: pitch 210Hz, rate 5.8)
# Author: Imran Ahmad
# =============================================================================
# Maps prosodic features to the VAD (Valence-Arousal-Dominance) model.
# Expected: pitch 210Hz + rate 5.8 sps → "angry" classification.
# =============================================================================

sentiment_result = sentiment_agent.analyze_sentiment(
    audio=synthetic_audio,
    emotion_scenario="angry",
)

print(f"\nPrimary emotion: {sentiment_result.primary_emotion}")
print(f"Confidence:      {sentiment_result.confidence}")
print(f"\nProsodic features:")
print(f"  Pitch:       {sentiment_result.prosodic_features.pitch_hz} Hz")
print(f"  Speech rate: {sentiment_result.prosodic_features.speech_rate_sps} sps")
print(f"  Energy:      {sentiment_result.prosodic_features.energy_db} dB")
print(f"\nVAD dimensions:")
print(f"  Valence:   {sentiment_result.valence}")
print(f"  Arousal:   {sentiment_result.arousal}")
print(f"  Dominance: {sentiment_result.dominance}")

---

# Part 3: Physical World Sensing Agents

## Smart Building Management Architecture
**Ref:** Chapter 11 — Smart Building Management Architecture

Physical world sensing agents bridge the gap between digital intelligence and physical environments. A smart building agent manages multiple **zones** (offices, server rooms, labs, meeting rooms), each with its own sensor array and control parameters.

The architecture follows the **Sense → Model → Plan → Act** loop:

1. **Sense** — IoT sensors stream temperature, humidity, CO2, occupancy, and light data.
2. **Model** — Sensor fusion aggregates readings over a temporal window to reduce noise.
3. **Plan** — Event detection uses pattern matching (lambda-based conditions) to identify anomalies. Control management applies proportional control with **deadband** (hysteresis) logic to prevent short-cycling.
4. **Act** — Issue control commands (cooling, ventilation, alerts) with computed intensity.

Key concepts:
- **Deadband / Hysteresis:** A tolerance zone around setpoints where no control action is taken, preventing oscillation.
- **Proportional Control:** Command intensity scales linearly with the deviation from setpoint.
- **Sensor Fusion:** Temporal averaging across a sliding window of readings to smooth transient spikes.

In [ ]:
# =============================================================================
# Cell 24: Import Layer — Physical World Backends
# Ref: Chapter 11 — Smart Building Management Architecture
# Author: Imran Ahmad
# =============================================================================

from datetime import datetime
from typing import Callable, Optional, Dict, Tuple
from collections import deque

if SIMULATION_MODE:
    from mock_backends import MockSensorStream, MockSensorReading
    logger.info("Physical backends: MockSensorStream loaded.")
else:
    logger.info("Physical backends: Live sensor interface loaded.")

In [ ]:
# =============================================================================
# Cell 25: Zone Configuration Data Structures
# Ref: Chapter 11 — Smart Building Management Architecture
# Author: Imran Ahmad
# =============================================================================

class ZoneType(Enum):
    """Classification of building zones.

    Ref: Smart Building Management Architecture.
    Author: Imran Ahmad
    """
    OFFICE = "office"
    SERVER_ROOM = "server_room"
    MEETING_ROOM = "meeting_room"
    LABORATORY = "laboratory"


@dataclass
class ZoneConfig:
    """Configuration parameters for a building zone.

    Ref: Smart Building Management Architecture — zone configuration.
    Author: Imran Ahmad

    Attributes:
        zone_id:           Unique zone identifier.
        zone_type:         Classification of the zone.
        temp_setpoint_f:   Target temperature in Fahrenheit.
        temp_deadband_f:   Deadband width (±) around setpoint.
        co2_threshold_ppm: CO2 level triggering ventilation.
        occupied_hours:    Tuple of (start_hour, end_hour) for normal occupancy.
        critical_temp_f:   Temperature triggering critical alert.
    """
    zone_id: str
    zone_type: ZoneType
    temp_setpoint_f: float = 72.0
    temp_deadband_f: float = 2.0
    co2_threshold_ppm: float = 1000.0
    occupied_hours: tuple = (8, 18)
    critical_temp_f: float = 85.0


@dataclass
class ZoneState:
    """Current processed state of a building zone after sensor fusion.

    Ref: Sensor Fusion Through Data Aggregation.
    Author: Imran Ahmad
    """
    zone_id: str
    avg_temperature_f: float
    avg_humidity_pct: float
    avg_co2_ppm: float
    occupancy_fraction: float
    light_lux: float
    timestamp: datetime = field(default_factory=datetime.now)
    readings_in_window: int = 1


@dataclass
class ControlCommand:
    """A control command issued by the building agent.

    Ref: Control Management and Feedback Loops.
    Author: Imran Ahmad
    """
    zone_id: str
    command_type: str       # 'cooling', 'heating', 'ventilation', 'alert'
    intensity_pct: float    # 0–100%
    reason: str
    timestamp: datetime = field(default_factory=datetime.now)


logger.success("Zone data structures defined: ZoneType, ZoneConfig, ZoneState, ControlCommand.")

In [ ]:
# =============================================================================
# Cell 26: EventPattern — Pattern Matching for Event Detection
# Ref: Chapter 11 — Event Detection Through Pattern Matching
# Author: Imran Ahmad
# =============================================================================

@dataclass
class EventPattern:
    """A named pattern with a lambda condition for event detection.

    Ref: Event Detection Through Pattern Matching.
    Author: Imran Ahmad

    The condition is a callable that receives (ZoneState, ZoneConfig)
    and returns True if the event is detected. This lambda-based approach
    allows flexible, composable event definitions without subclassing.

    Attributes:
        name:      Human-readable event name.
        severity:  'info', 'warning', or 'critical'.
        condition: Callable(ZoneState, ZoneConfig) -> bool.
        message:   Template string with {placeholders} for state values.
    """
    name: str
    severity: str  # 'info', 'warning', 'critical'
    condition: Callable  # (ZoneState, ZoneConfig) -> bool
    message: str


# Define event patterns from the chapter
# Ref: Event Detection Through Pattern Matching — pattern definitions
EVENT_PATTERNS: List[EventPattern] = [
    EventPattern(
        name="critical_temp",
        severity="critical",
        condition=lambda state, config: state.avg_temperature_f >= config.critical_temp_f,
        message="CRITICAL temp in {zone_id}: {temp:.1f}°F (threshold: {threshold:.1f}°F)",
    ),
    EventPattern(
        name="high_co2",
        severity="warning",
        condition=lambda state, config: state.avg_co2_ppm > config.co2_threshold_ppm,
        message="High CO2 in {zone_id}: {co2:.0f} ppm (threshold: {co2_threshold:.0f} ppm)",
    ),
    EventPattern(
        name="after_hours_occupancy",
        severity="warning",
        condition=lambda state, config: (
            state.occupancy_fraction > 0.1
            and (state.timestamp.hour < config.occupied_hours[0]
                 or state.timestamp.hour >= config.occupied_hours[1])
        ),
        message="Unexpected occupancy in {zone_id} outside hours "
                "(occ: {occ:.0%}, hour: {hour})",
    ),
]

logger.success(f"Event patterns defined: {len(EVENT_PATTERNS)} patterns "
               f"({', '.join(p.name for p in EVENT_PATTERNS)})")

In [ ]:
# =============================================================================
# Cell 27: ControlManager — Proportional Control with Deadband
# Ref: Chapter 11 — Control Management and Feedback Loops
# Author: Imran Ahmad
# =============================================================================

class ControlManager:
    """Manages proportional control commands with deadband logic.

    Ref: Chapter 11 — Control Management and Feedback Loops.
    Author: Imran Ahmad

    The deadband (hysteresis) prevents short-cycling: if the measured
    value is within ±deadband of the setpoint, no control action is taken.
    Outside the deadband, command intensity scales proportionally with
    deviation from the setpoint.

    Proportional control formula:
        intensity = min(100, (deviation / max_deviation) * 100)
    """

    def __init__(self) -> None:
        self.logger = AgentLogger(name="ControlMgr")

    def compute_temperature_command(
        self, state: ZoneState, config: ZoneConfig
    ) -> Optional[ControlCommand]:
        """Compute temperature control command with deadband.

        Ref: Control Management and Feedback Loops — proportional control.
        Author: Imran Ahmad

        Args:
            state:  Current zone state (post-sensor-fusion).
            config: Zone configuration with setpoint and deadband.

        Returns:
            ControlCommand if action needed, None if within deadband.
        """
        deviation = state.avg_temperature_f - config.temp_setpoint_f

        # Deadband check: no action if within ± deadband
        if abs(deviation) <= config.temp_deadband_f:
            return None

        # Proportional control: scale intensity with deviation
        max_deviation = config.critical_temp_f - config.temp_setpoint_f
        intensity = min(100.0, abs(deviation) / max_deviation * 100)

        if deviation > 0:
            cmd_type = "cooling"
            reason = (f"Temperature {state.avg_temperature_f:.1f}°F exceeds "
                      f"setpoint {config.temp_setpoint_f:.1f}°F + "
                      f"deadband {config.temp_deadband_f:.1f}°F")
        else:
            cmd_type = "heating"
            reason = (f"Temperature {state.avg_temperature_f:.1f}°F below "
                      f"setpoint {config.temp_setpoint_f:.1f}°F - "
                      f"deadband {config.temp_deadband_f:.1f}°F")

        cmd = ControlCommand(
            zone_id=state.zone_id,
            command_type=cmd_type,
            intensity_pct=round(intensity, 1),
            reason=reason,
        )
        self.logger.success(
            f"{cmd_type.capitalize()} command issued: intensity {cmd.intensity_pct}%"
        )
        return cmd

    def compute_ventilation_command(
        self, state: ZoneState, config: ZoneConfig
    ) -> Optional[ControlCommand]:
        """Compute ventilation command based on CO2 levels.

        Ref: Control Management and Feedback Loops — ventilation control.
        Author: Imran Ahmad

        Args:
            state:  Current zone state.
            config: Zone configuration with CO2 threshold.

        Returns:
            ControlCommand if CO2 exceeds threshold, None otherwise.
        """
        if state.avg_co2_ppm <= config.co2_threshold_ppm:
            return None

        # Proportional: scale intensity with CO2 excess
        excess = state.avg_co2_ppm - config.co2_threshold_ppm
        max_excess = 1000.0  # ppm above threshold for 100% intensity
        intensity = min(100.0, (excess / max_excess) * 100)

        cmd = ControlCommand(
            zone_id=state.zone_id,
            command_type="ventilation",
            intensity_pct=round(intensity, 1),
            reason=f"CO2 {state.avg_co2_ppm:.0f} ppm exceeds threshold {config.co2_threshold_ppm:.0f} ppm",
        )
        self.logger.success(
            f"Ventilation command issued: intensity {cmd.intensity_pct}%"
        )
        return cmd

In [ ]:
# =============================================================================
# Cell 28: SmartBuildingAgent — Full Sense-Model-Plan-Act Loop
# Ref: Chapter 11 — Smart Building Management Architecture,
#      Event Detection, Control Management, Sensor Fusion
# Author: Imran Ahmad
# =============================================================================

class SmartBuildingAgent:
    """Agent that manages smart building zones with sensor fusion,
    event detection, and proportional control.

    Ref: Chapter 11 — Smart Building Management Architecture.
    Author: Imran Ahmad

    Integrates:
    - Sensor fusion (temporal averaging over a sliding window)
    - Event detection (lambda-based pattern matching)
    - Control management (proportional control with deadband)

    Args:
        sensor_stream:   A sensor backend (real or mock) with .get_reading()/.get_history().
        zone_configs:    Dict mapping zone_id to ZoneConfig.
        event_patterns:  List of EventPattern for anomaly detection.
        fusion_window:   Number of readings for temporal averaging.
    """

    def __init__(
        self,
        sensor_stream,
        zone_configs: Dict[str, ZoneConfig],
        event_patterns: List[EventPattern],
        fusion_window: int = 5,
    ) -> None:
        self.sensor_stream = sensor_stream
        self.zone_configs = zone_configs
        self.event_patterns = event_patterns
        self.fusion_window = fusion_window
        self.control_manager = ControlManager()
        self.logger = AgentLogger(name="BuildingAgent")
        self.logger.info("Initializing Smart Building Agent...")
        mode_label = "Simulation Mode" if SIMULATION_MODE else "Live Mode"
        self.logger.success(
            f"Agent ready ({mode_label}). "
            f"Zones: {len(zone_configs)}, Patterns: {len(event_patterns)}"
        )

    def _fuse_readings(self, zone_id: str) -> ZoneState:
        """Apply sensor fusion via temporal averaging.

        Ref: Sensor Fusion Through Data Aggregation.
        Author: Imran Ahmad

        Averages temperature, humidity, CO2, occupancy, and light
        over the sliding window to reduce transient noise.

        Args:
            zone_id: The zone to fuse readings for.

        Returns:
            ZoneState with averaged sensor values.
        """
        history = self.sensor_stream.get_history(zone_id, window_size=self.fusion_window)
        n = len(history)

        avg_temp = sum(r.temperature_f for r in history) / n
        avg_humidity = sum(r.humidity_pct for r in history) / n
        avg_co2 = sum(r.co2_ppm for r in history) / n
        avg_occ = sum(r.occupancy_fraction for r in history) / n
        avg_light = sum(r.light_lux for r in history) / n

        return ZoneState(
            zone_id=zone_id,
            avg_temperature_f=round(avg_temp, 1),
            avg_humidity_pct=round(avg_humidity, 1),
            avg_co2_ppm=round(avg_co2, 1),
            occupancy_fraction=round(avg_occ, 2),
            light_lux=round(avg_light, 1),
            timestamp=history[-1].timestamp,
            readings_in_window=n,
        )

    def _detect_events(
        self, state: ZoneState, config: ZoneConfig
    ) -> List[str]:
        """Run all event patterns against the current state.

        Ref: Event Detection Through Pattern Matching.
        Author: Imran Ahmad

        Args:
            state:  Current zone state (post-fusion).
            config: Zone configuration.

        Returns:
            List of triggered event messages.
        """
        alerts = []
        for pattern in self.event_patterns:
            try:
                if pattern.condition(state, config):
                    msg = pattern.message.format(
                        zone_id=state.zone_id,
                        temp=state.avg_temperature_f,
                        threshold=config.critical_temp_f,
                        co2=state.avg_co2_ppm,
                        co2_threshold=config.co2_threshold_ppm,
                        occ=state.occupancy_fraction,
                        hour=state.timestamp.hour,
                    )
                    if pattern.severity == "critical":
                        self.logger.error(msg)
                    else:
                        self.logger.error(msg)  # warnings also red for visibility
                    alerts.append(f"[{pattern.severity.upper()}] {msg}")
            except Exception:
                pass  # Guard against malformed patterns
        return alerts

    @graceful_fallback(
        fallback_value={"alerts": [], "commands": [], "state": None},
        chapter_ref="Smart Building Management Architecture",
    )
    def process_zone(self, zone_id: str) -> Dict:
        """Full Sense-Model-Plan-Act cycle for a single zone.

        Ref: Smart Building Management Architecture — main loop.
        Author: Imran Ahmad

        Sense:  Read sensors, fuse over temporal window.
        Model:  Detect events via pattern matching.
        Plan:   Compute control commands with deadband logic.
        Act:    Return alerts, commands, and fused state.

        Args:
            zone_id: The zone identifier to process.

        Returns:
            Dict with 'alerts', 'commands', and 'state'.
        """
        config = self.zone_configs.get(zone_id)
        if config is None:
            self.logger.error(f"Unknown zone: {zone_id}")
            return {"alerts": [], "commands": [], "state": None}

        self.logger.info(f"process_zone called: {zone_id} ({config.zone_type.value})")

        # Sense: get fresh reading + fuse history
        self.sensor_stream.get_reading(zone_id)
        state = self._fuse_readings(zone_id)

        # Model: detect events
        alerts = self._detect_events(state, config)

        # Plan: compute control commands
        commands = []
        temp_cmd = self.control_manager.compute_temperature_command(state, config)
        if temp_cmd:
            commands.append(temp_cmd)
        vent_cmd = self.control_manager.compute_ventilation_command(state, config)
        if vent_cmd:
            commands.append(vent_cmd)

        # Act: log summary
        self.logger.success(
            f"process_zone completed. {len(alerts)} alerts, "
            f"{len(commands)} commands"
            f"{' (within deadband)' if not commands and not alerts else ''}."
        )

        return {"alerts": alerts, "commands": commands, "state": state}

In [ ]:
# =============================================================================
# Cell 29: Zone Configuration & Agent Initialization
# Ref: Chapter 11 — Smart Building Management Architecture
# Author: Imran Ahmad
# =============================================================================

zone_configs = {
    "zone_a_office": ZoneConfig(
        zone_id="zone_a_office",
        zone_type=ZoneType.OFFICE,
        temp_setpoint_f=72.0,
        temp_deadband_f=2.0,
        co2_threshold_ppm=1000.0,
        occupied_hours=(8, 18),
        critical_temp_f=85.0,
    ),
    "zone_b_meeting": ZoneConfig(
        zone_id="zone_b_meeting",
        zone_type=ZoneType.MEETING_ROOM,
        temp_setpoint_f=70.0,
        temp_deadband_f=2.0,
        co2_threshold_ppm=1000.0,
        occupied_hours=(8, 20),
        critical_temp_f=85.0,
    ),
    "zone_c_lab": ZoneConfig(
        zone_id="zone_c_lab",
        zone_type=ZoneType.LABORATORY,
        temp_setpoint_f=68.0,
        temp_deadband_f=1.5,
        co2_threshold_ppm=1000.0,
        occupied_hours=(7, 22),
        critical_temp_f=80.0,
    ),
    "zone_d_server": ZoneConfig(
        zone_id="zone_d_server",
        zone_type=ZoneType.SERVER_ROOM,
        temp_setpoint_f=65.0,
        temp_deadband_f=1.0,
        co2_threshold_ppm=800.0,
        occupied_hours=(0, 24),  # always monitored
        critical_temp_f=80.0,
    ),
}

if SIMULATION_MODE:
    sensor_stream = MockSensorStream()
else:
    # In Live Mode, initialize real sensor interface here
    pass

building_agent = SmartBuildingAgent(
    sensor_stream=sensor_stream,
    zone_configs=zone_configs,
    event_patterns=EVENT_PATTERNS,
    fusion_window=5,
)

## Building Agent Scenario Demos

The following cells demonstrate four scenarios from the chapter, each exercising different aspects of the Sense → Model → Plan → Act loop:

| # | Scenario | Zone | Expected Behavior |
|---|---|---|---|
| 1 | Normal office | zone_a_office | 72°F within deadband → no commands |
| 2 | Server room overheat | zone_d_server | 96.5°F → critical alert + max cooling |
| 3 | After-hours intrusion | zone_b_meeting | Occupancy 0.9 at 23:00 → warning |
| 4 | High CO2 lab | zone_c_lab | 1350 ppm → ventilation command |

**Ref:** Event Detection Through Pattern Matching, Control Management and Feedback Loops

In [ ]:
# =============================================================================
# Cell 31: Scenario 1 — Normal Office (No Action)
# Ref: Chapter 11 — Smart Building Management Architecture
# Author: Imran Ahmad
# =============================================================================
# zone_a_office: 72°F is exactly at setpoint, well within the ±2°F deadband.
# Expected: 0 alerts, 0 commands — the building is operating normally.
# =============================================================================

print("=" * 60)
print("SCENARIO 1: Normal Office — zone_a_office")
print("=" * 60)
result_normal = building_agent.process_zone("zone_a_office")
print(f"\nAlerts:   {len(result_normal['alerts'])}")
print(f"Commands: {len(result_normal['commands'])}")
if result_normal['state']:
    s = result_normal['state']
    print(f"\nFused state: temp={s.avg_temperature_f}°F, "
          f"humidity={s.avg_humidity_pct}%, "
          f"CO2={s.avg_co2_ppm}ppm, "
          f"occupancy={s.occupancy_fraction:.0%}")

In [ ]:
# =============================================================================
# Cell 32: Scenario 2 — Server Room Overheat (Critical Alert + Cooling)
# Ref: Chapter 11 — Event Detection Through Pattern Matching
# Author: Imran Ahmad
# =============================================================================
# zone_d_server: 96.5°F far exceeds critical threshold of 80°F.
# Expected: RED critical_temp alert + cooling command at high intensity.
# =============================================================================

print("=" * 60)
print("SCENARIO 2: Server Room Overheat — zone_d_server")
print("=" * 60)
result_overheat = building_agent.process_zone("zone_d_server")
print(f"\nAlerts:   {len(result_overheat['alerts'])}")
for alert in result_overheat['alerts']:
    print(f"  {alert}")
print(f"Commands: {len(result_overheat['commands'])}")
for cmd in result_overheat['commands']:
    print(f"  {cmd.command_type}: intensity={cmd.intensity_pct}% — {cmd.reason}")

In [ ]:
# =============================================================================
# Cell 33: Scenario 3 — After-Hours Intrusion (Warning)
# Ref: Chapter 11 — Event Detection Through Pattern Matching
# Author: Imran Ahmad
# =============================================================================
# zone_b_meeting: occupancy 0.9 at 23:00 — outside occupied_hours (8-20).
# Expected: RED warning alert for unexpected after-hours occupancy.
# =============================================================================

print("=" * 60)
print("SCENARIO 3: After-Hours Intrusion — zone_b_meeting")
print("=" * 60)
result_intrusion = building_agent.process_zone("zone_b_meeting")
print(f"\nAlerts:   {len(result_intrusion['alerts'])}")
for alert in result_intrusion['alerts']:
    print(f"  {alert}")
print(f"Commands: {len(result_intrusion['commands'])}")

In [ ]:
# =============================================================================
# Cell 34: Scenario 4 — High CO2 Lab (Ventilation Command)
# Ref: Chapter 11 — Control Management and Feedback Loops
# Author: Imran Ahmad
# =============================================================================
# zone_c_lab: CO2 at 1350 ppm, threshold is 1000 ppm.
# Expected: Ventilation command with proportional intensity.
# Intensity = (1350 - 1000) / 1000 * 100 = 35%
# =============================================================================

print("=" * 60)
print("SCENARIO 4: High CO2 Lab — zone_c_lab")
print("=" * 60)
result_co2 = building_agent.process_zone("zone_c_lab")
print(f"\nAlerts:   {len(result_co2['alerts'])}")
for alert in result_co2['alerts']:
    print(f"  {alert}")
print(f"Commands: {len(result_co2['commands'])}")
for cmd in result_co2['commands']:
    print(f"  {cmd.command_type}: intensity={cmd.intensity_pct}% — {cmd.reason}")

---

# Part 4: Cross-Domain Summary

## Multi-Modal Agent Comparison
**Ref:** Chapter 11 — Summary

The following table compares the three agent domains covered in this chapter across their key architectural dimensions:

| Dimension | Vision-Language Agent | Audio Processing Agent | Physical World Agent |
|---|---|---|---|
| **Input Type** | RGB images (camera, file) | Audio waveforms (microphone, file) | IoT sensor streams (temp, CO2, occupancy) |
| **Encoding Method** | CLIP ViT-L/14 vision encoder → patch embeddings | Mel-spectrogram → Whisper encoder | Raw numeric readings → temporal averaging |
| **Alignment Strategy** | Linear projection layer mapping visual features to language embedding space | Implicit alignment via encoder-decoder attention in Whisper | Sensor fusion through sliding window averaging |
| **Reasoning** | Chain-of-thought in language model conditioned on visual tokens | Post-processing (filler removal) + prosodic feature mapping to VAD model | Lambda-based event pattern matching + proportional control with deadband |
| **Action Type** | Natural-language answer (VQAResult) | Structured transcript (TranscriptionResult) / emotion label (SentimentResult) | Control commands (cooling, ventilation) + alert notifications |
| **Error Strategy** | @graceful_fallback returns safe VQAResult | @graceful_fallback returns empty TranscriptionResult | @graceful_fallback returns empty dict with state=None |
| **Chapter Section** | Architecture of VL Agents, Building a VQA Agent, Integration Patterns | Architecture of Audio Agents, Speech Recognition, Voice Sentiment | Smart Building Architecture, Event Detection, Control Management, Sensor Fusion |

### Key Takeaways

1. **Unified Loop:** All three agents share the same **Sense → Model → Plan → Act** loop, despite operating on fundamentally different modalities.
2. **Alignment is the Core Challenge:** Each modality requires a different strategy to bridge the gap between raw sensor data and actionable representations — linear projection for vision, encoder-decoder attention for audio, and temporal averaging for physical sensors.
3. **Resilience by Design:** The `@graceful_fallback` decorator pattern ensures that no single agent failure cascades into a notebook crash, enabling safe experimentation across all three domains.

In [ ]:
# =============================================================================
# Cell 36: Chapter 11 Completion Banner
# Ref: Chapter 11 — Summary
# Author: Imran Ahmad
# =============================================================================

from agent_logger import AgentLogger

final_logger = AgentLogger(name="Chapter11")

print("=" * 65)
print("  CHAPTER 11 — MULTI-MODAL PERCEPTION AGENTS — COMPLETE")
print("=" * 65)
print()
print("  Agents executed:")
print("    1. VisionQuestionAnsweringAgent  — 3 queries + 1 error demo")
print("    2. SpeechRecognitionAgent        — CLEAN + VERBATIM modes")
print("    3. VoiceSentimentAgent           — Prosodic → VAD mapping")
print("    4. SmartBuildingAgent            — 4 zone scenarios")
print()
print("  All agents used the Sense → Model → Plan → Act loop.")
print("  All outputs were color-coded via AgentLogger.")
print("  All inference methods protected by @graceful_fallback.")
print()

final_logger.info(
    "Chapter 11 complete. All agents executed successfully. "
    "Book: 30 Agents Every AI Engineer Must Build — Imran Ahmad."
)